# Recurrent networks: memory for sequences
**Machine Learning · Week 9 (Deep learning: images, sequences and attention) · Day 4**

Lesson page: https://sara-aminpour.com/tutorials/ml/w9d4.html

A **recurrent neural network** (RNN) reads a sequence one step at a time and carries a **hidden state** $h$,
its memory, from step to step: $h_t = \tanh(W h_{t-1} + U x_t + b)$. The same weights at every step,
like a CNN's filters at every position. Here is one, by hand, with a 2-number memory.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

torch.set_num_threads(1)
W = np.array([[0.9, 0.0], [0.5, 0.5]])     # memory -> memory
U = np.array([[1.0], [0.0]])                # input  -> memory
xs = [1.0, 0.0, 0.0, 0.0, -1.0, 0.0, 0.0]   # a short sequence
h = np.zeros(2)
for t, x in enumerate(xs):
    h = np.tanh(W @ h + U @ [x])
    print(f"step {t}: input {x:+.0f} -> memory {np.round(h, 3)}")

## Can it remember? The first-number task
Each sequence is $T$ random numbers. The label is 1 if the *first* number was positive, else 0. The network
reads the whole sequence and only answers at the end, so it must carry one bit of information through $T$
steps while ignoring everything after it. A plain RNN against an **LSTM** (long short-term memory), whose
memory cell is protected by gates. We start the LSTM's forget gate open (bias 1), a standard trick.

In [ ]:
class Reader(nn.Module):
    def __init__(self, kind, hidden=16):
        super().__init__()
        torch.manual_seed(0)
        self.rnn = (nn.RNN if kind == "RNN" else nn.LSTM)(1, hidden, batch_first=True)
        self.out = nn.Linear(hidden, 2)
        if kind == "LSTM":
            with torch.no_grad():                                 # the gates are stacked as input, forget, cell, output
                self.rnn.bias_ih_l0[hidden:2 * hidden].fill_(1.0)
                self.rnn.bias_hh_l0[hidden:2 * hidden].fill_(1.0)

    def forward(self, x):                      # x: (batch, T, 1)
        states, _ = self.rnn(x)                # the memory after every step: (batch, T, hidden)
        return self.out(states[:, -1])         # answer from the last one

def sequences(n, T, gen):
    x = torch.randn(n, T, 1, generator=gen)
    return x, (x[:, 0, 0] > 0).long()

def train_and_test(kind, T, steps=1500):
    model = Reader(kind)
    opt = torch.optim.Adam(model.parameters(), lr=3e-3)
    loss_fn = nn.CrossEntropyLoss()
    gen = torch.Generator().manual_seed(0)
    for _ in range(steps):
        x, y = sequences(64, T, gen)
        opt.zero_grad()
        loss_fn(model(x), y).backward()
        opt.step()
    x, y = sequences(2000, T, torch.Generator().manual_seed(1))
    with torch.no_grad():
        return (model(x).argmax(dim=1) == y).float().mean().item()

t0 = time.perf_counter()
print(f"{'length T':>9s} {'RNN':>7s} {'LSTM':>7s}")
for T in [5, 20, 50]:
    print(f"{T:9d} {train_and_test('RNN', T):7.3f} {train_and_test('LSTM', T):7.3f}")
print(f"({time.perf_counter() - t0:.0f} s; 0.5 = guessing)")

## Why the plain RNN forgets: vanishing gradients
To learn from step 1, the error at the end must flow back through every step, and each step multiplies it
by $W$ and by the slope of tanh. For an untrained network with $T = 50$, how strongly does the output depend
on each input? (Gradient size, on a log scale.)

In [ ]:
x = torch.randn(1, 50, 1, generator=torch.Generator().manual_seed(2), requires_grad=True)
plt.figure(figsize=(6.4, 3.4))
for kind, color in [("RNN", "#ff8a50"), ("LSTM", "#17a98c")]:
    x.grad = None
    Reader(kind)(x)[0, 1].backward()
    plt.semilogy(range(1, 51), x.grad[0, :, 0].abs().numpy() + 1e-12, color=color, label=kind)
plt.xlabel("input step")
plt.ylabel("effect on the answer")
plt.legend()
plt.title("How far back can the answer see?")
plt.show()

## Your turn: where does the plain RNN give up?
Train the plain RNN on lengths 8, 10, 12, 15 and 20 and print its test accuracy for each.
Between which lengths does it stop remembering?

In [ ]:
for T in [8, 10, 12, 15, 20]:
    acc = None   # TODO: train_and_test("RNN", T)
    print(f"T = {T:2d}: {acc}")

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
for T in [8, 10, 12, 15, 20]:
    print(f"T = {T:2d}: {train_and_test('RNN', T):.3f}")
print("It still remembers 15 steps back, but not 20: somewhere in between, the gradient from the answer")
print("back to step 1 becomes too small to learn from. The LSTM above managed 20 easily.")